<a href="https://colab.research.google.com/github/ranjetmahato416/-Lung-CT-Image-Classification-Using-Public-Medical-Imaging-Data-/blob/main/Notebook/Notebook_16B_Dedicated_Supervised_Input_Domain_Validator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 16B — Dedicated Supervised Input Domain Validator

## Objective

The final DenseNet121 lung nodule classifier is a closed-set binary
classifier and therefore produces a Benign or Malignant prediction for
any image supplied to it.

Notebook 16 demonstrated that a DenseNet121 feature-space k-NN OOD
detector could reject clearly unrelated natural images, MRI and
ultrasound images. However, it accepted most chest X-rays and
disproportionately rejected legitimate malignant CT nodules.

This notebook therefore evaluates a dedicated supervised input-domain
classifier.

The domain validator performs only one task:

- Supported: preprocessed lung CT nodule image compatible with the
  classifier's training domain.
- Unsupported: image outside the supported input domain.

The domain validator does not classify malignancy and does not determine
medical-image modality with clinical certainty.

Only images accepted by this validator will later be forwarded to the
DenseNet121 Benign/Malignant classifier.

##2. Imports and reproducibility

In [51]:
from pathlib import Path
import json
import os
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    roc_curve,
    precision_score,
    recall_score,
    f1_score
)

from sklearn.utils.class_weight import (
    compute_class_weight
)

In [52]:
SEED = 42

random.seed(
    SEED
)

np.random.seed(
    SEED
)

tf.random.set_seed(
    SEED
)

print(
    "TensorFlow:",
    tf.__version__
)

TensorFlow: 2.20.0


##3. Mount Google Drive

In [53]:
from google.colab import drive

MOUNT_POINT = (
    "/content/drive"
)

if os.path.ismount(
    MOUNT_POINT
):

    print(
        "Google Drive already mounted."
    )

else:

    drive.mount(
        MOUNT_POINT
    )

Google Drive already mounted.


##4. Define paths

In [54]:
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Dissertation"
)


PROCESSED_ROOT = (
    PROJECT_ROOT
    / "Dataset"
    / "Processed"
)


CNN_DATASET_ROOT = (
    PROCESSED_ROOT
    / "CNN_Dataset"
)


OOD_ROOT = (
    PROJECT_ROOT
    / "OOD_Evaluation"
)


DOMAIN_MODEL_ROOT = (
    PROJECT_ROOT
    / "Models"
    / "Domain_Validation"
    / "Experiment_02_Supervised"
)


DOMAIN_MODEL_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


print(
    "CT dataset:",
    CNN_DATASET_ROOT
)

print(
    "OOD root:",
    OOD_ROOT
)

print(
    "Output:",
    DOMAIN_MODEL_ROOT
)

CT dataset: /content/drive/MyDrive/Dissertation/Dataset/Processed/CNN_Dataset
OOD root: /content/drive/MyDrive/Dissertation/OOD_Evaluation
Output: /content/drive/MyDrive/Dissertation/Models/Domain_Validation/Experiment_02_Supervised


##5. Load the existing CT patient-level splits

In [55]:
TRAIN_CSV = (
    PROCESSED_ROOT
    / "train.csv"
)

VALIDATION_CSV = (
    PROCESSED_ROOT
    / "validation.csv"
)

TEST_CSV = (
    PROCESSED_ROOT
    / "test.csv"
)


train_ct_df = pd.read_csv(
    TRAIN_CSV
)

validation_ct_df = pd.read_csv(
    VALIDATION_CSV
)

test_ct_df = pd.read_csv(
    TEST_CSV
)


print(
    "Train CT:",
    len(train_ct_df)
)

print(
    "Validation CT:",
    len(validation_ct_df)
)

print(
    "Test CT:",
    len(test_ct_df)
)

Train CT: 1149
Validation CT: 237
Test CT: 256


###5.1 Resolve colab image path

In [56]:
def resolve_image_path(row):

    label_name = (
        "Malignant"
        if int(row["CancerLabel"]) == 1
        else "Benign"
    )

    original_path = Path(
        str(row["ImagePath"])
    )

    filename = (
        original_path.name
    )

    resolved_path = (
        CNN_DATASET_ROOT
        / label_name
        / filename
    )

    return str(
        resolved_path
    )

In [57]:
for df in [
    train_ct_df,
    validation_ct_df,
    test_ct_df
]:

    df[
        "ResolvedImagePath"
    ] = df.apply(
        resolve_image_path,
        axis=1
    )

In [58]:
#Verify#

for name, df in [
    (
        "Train",
        train_ct_df
    ),
    (
        "Validation",
        validation_ct_df
    ),
    (
        "Test",
        test_ct_df
    )
]:

    exists = (
        df[
            "ResolvedImagePath"
        ]
        .apply(
            lambda path:
                Path(path).exists()
        )
    )

    print(
        name,
        ":",
        exists.sum(),
        "/",
        len(df)
    )

Train : 1149 / 1149
Validation : 237 / 237
Test : 256 / 256


##6. Discover the medical OOD images

In [59]:
SUPPORTED_EXTENSIONS = {
    ".png",
    ".jpg",
    ".jpeg"
}


def collect_images(
    directory,
    source_name
):

    directory = Path(
        directory
    )

    paths = sorted(
        [
            path

            for path
            in directory.rglob(
                "*"
            )

            if (
                path.is_file()

                and

                path.suffix.lower()
                in SUPPORTED_EXTENSIONS
            )
        ]
    )


    dataframe = pd.DataFrame(
        {
            "ImagePath":
                [
                    str(path)
                    for path
                    in paths
                ],

            "Source":
                source_name
        }
    )

    return dataframe

In [60]:
xray_df = collect_images(
    OOD_ROOT
    / "Chest_Xray",
    "Chest_Xray"
)


mri_df = collect_images(
    OOD_ROOT
    / "MRI",
    "MRI"
)


ultrasound_df = collect_images(
    OOD_ROOT
    / "Ultrasound",
    "Ultrasound"
)


print(
    "Chest X-rays:",
    len(xray_df)
)

print(
    "MRI:",
    len(mri_df)
)

print(
    "Ultrasound:",
    len(ultrasound_df)
)

Chest X-rays: 57
MRI: 80
Ultrasound: 55


##7. Split each medical OOD category independently

- 60% train
- 20% validation
- 20% test

In [61]:
def split_ood_source(
    dataframe
):

    train_part, temp_part = (
        train_test_split(
            dataframe,
            test_size=0.40,
            random_state=SEED,
            shuffle=True
        )
    )


    validation_part, test_part = (
        train_test_split(
            temp_part,
            test_size=0.50,
            random_state=SEED,
            shuffle=True
        )
    )


    train_part = (
        train_part.copy()
    )

    validation_part = (
        validation_part.copy()
    )

    test_part = (
        test_part.copy()
    )


    train_part[
        "Split"
    ] = "train"

    validation_part[
        "Split"
    ] = "validation"

    test_part[
        "Split"
    ] = "test"


    return (
        train_part,
        validation_part,
        test_part
    )

In [62]:
(
    xray_train,
    xray_validation,
    xray_test
) = split_ood_source(
    xray_df
)


(
    mri_train,
    mri_validation,
    mri_test
) = split_ood_source(
    mri_df
)


(
    ultrasound_train,
    ultrasound_validation,
    ultrasound_test
) = split_ood_source(
    ultrasound_df
)

In [63]:
for name, (
    train_part,
    validation_part,
    test_part
) in {

    "Chest X-ray":
        (
            xray_train,
            xray_validation,
            xray_test
        ),

    "MRI":
        (
            mri_train,
            mri_validation,
            mri_test
        ),

    "Ultrasound":
        (
            ultrasound_train,
            ultrasound_validation,
            ultrasound_test
        )

}.items():

    print(
        "\n",
        name
    )

    print(
        "Train:",
        len(train_part)
    )

    print(
        "Validation:",
        len(validation_part)
    )

    print(
        "Test:",
        len(test_part)
    )


 Chest X-ray
Train: 34
Validation: 11
Test: 12

 MRI
Train: 48
Validation: 16
Test: 16

 Ultrasound
Train: 33
Validation: 11
Test: 11


##8. Add natural-image OOD samples

In [64]:
(
    cifar_train_images,
    _
), (
    cifar_test_images,
    _
) = (
    tf.keras.datasets
    .cifar10
    .load_data()
)

NATURAL_TRAIN_COUNT = 90
NATURAL_VALIDATION_COUNT = 30
NATURAL_TEST_COUNT = 30

####Create a storage folder

In [65]:
CIFAR_EXPORT_ROOT = (
    OOD_ROOT
    / "Natural_CIFAR"
)

CIFAR_EXPORT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

In [66]:
# Helper #

from PIL import Image


def save_cifar_images(
    images,
    start_index,
    count,
    split_name
):

    split_directory = (
        CIFAR_EXPORT_ROOT
        / split_name
    )

    split_directory.mkdir(
        parents=True,
        exist_ok=True
    )


    records = []


    for i in range(
        count
    ):

        source_index = (
            start_index
            + i
        )

        output_path = (
            split_directory
            / (
                f"cifar_"
                f"{source_index:05d}"
                f".png"
            )
        )


        if not output_path.exists():

            Image.fromarray(
                images[
                    source_index
                ]
            ).save(
                output_path
            )


        records.append(
            {
                "ImagePath":
                    str(
                        output_path
                    ),

                "Source":
                    "Natural_CIFAR",

                "Split":
                    split_name
            }
        )


    return pd.DataFrame(
        records
    )

In [67]:
natural_train = (
    save_cifar_images(
        cifar_train_images,
        start_index=0,
        count=NATURAL_TRAIN_COUNT,
        split_name="train"
    )
)


natural_validation = (
    save_cifar_images(
        cifar_train_images,
        start_index=1000,
        count=NATURAL_VALIDATION_COUNT,
        split_name="validation"
    )
)


natural_test = (
    save_cifar_images(
        cifar_test_images,
        start_index=0,
        count=NATURAL_TEST_COUNT,
        split_name="test"
    )
)

##9. Build the unsupported DataFrames

In [68]:
ood_train_df = pd.concat(
    [
        xray_train,
        mri_train,
        ultrasound_train,
        natural_train
    ],
    ignore_index=True
)


ood_validation_df = pd.concat(
    [
        xray_validation,
        mri_validation,
        ultrasound_validation,
        natural_validation
    ],
    ignore_index=True
)


ood_test_df = pd.concat(
    [
        xray_test,
        mri_test,
        ultrasound_test,
        natural_test
    ],
    ignore_index=True
)

In [69]:
print(
    "OOD train:",
    len(ood_train_df)
)

print(
    "OOD validation:",
    len(ood_validation_df)
)

print(
    "OOD test:",
    len(ood_test_df)
)


display(
    ood_train_df[
        "Source"
    ]
    .value_counts()
)

OOD train: 205
OOD validation: 68
OOD test: 69


,count
Source,
Natural_CIFAR,90
MRI,48
Chest_Xray,34
Ultrasound,33


##10. Build supported CT DataFrames

In [70]:
Supported = 1
Unsupported = 0

In [71]:
def prepare_ct_domain_df(
    dataframe,
    split_name
):

    result = pd.DataFrame(
        {
            "ImagePath":
                dataframe[
                    "ResolvedImagePath"
                ]
                .values,

            "Source":
                "Lung_CT_Nodule",

            "CancerLabel":
                dataframe[
                    "CancerLabel"
                ]
                .values,

            "PatientID":
                dataframe[
                    "PatientID"
                ]
                .values,

            "Split":
                split_name,

            "DomainLabel":
                1
        }
    )

    return result

In [72]:
ct_train_domain = (
    prepare_ct_domain_df(
        train_ct_df,
        "train"
    )
)


ct_validation_domain = (
    prepare_ct_domain_df(
        validation_ct_df,
        "validation"
    )
)


ct_test_domain = (
    prepare_ct_domain_df(
        test_ct_df,
        "test"
    )
)

####Add unsupported labels

In [73]:
for dataframe in [
    ood_train_df,
    ood_validation_df,
    ood_test_df
]:

    dataframe[
        "DomainLabel"
    ] = 0

    dataframe[
        "CancerLabel"
    ] = np.nan

    dataframe[
        "PatientID"
    ] = np.nan

##11. Build final train/validation/test manifests

In [74]:
domain_train_df = pd.concat(
    [
        ct_train_domain,
        ood_train_df
    ],
    ignore_index=True
)


domain_validation_df = pd.concat(
    [
        ct_validation_domain,
        ood_validation_df
    ],
    ignore_index=True
)


domain_test_df = pd.concat(
    [
        ct_test_domain,
        ood_test_df
    ],
    ignore_index=True
)

####Suffle training only:

In [75]:
domain_train_df = (
    domain_train_df
    .sample(
        frac=1,
        random_state=SEED
    )
    .reset_index(
        drop=True
    )
)

##12. Verify distributions before training

In [76]:
for name, df in [
    (
        "TRAIN",
        domain_train_df
    ),
    (
        "VALIDATION",
        domain_validation_df
    ),
    (
        "TEST",
        domain_test_df
    )
]:

    print(
        "\n",
        "=" * 60
    )

    print(
        name
    )

    print(
        "=" * 60
    )

    print(
        "Total:",
        len(df)
    )

    print(
        "\nDomain labels:"
    )

    print(
        df[
            "DomainLabel"
        ]
        .value_counts()
    )

    print(
        "\nSources:"
    )

    print(
        df[
            "Source"
        ]
        .value_counts()
    )


TRAIN
Total: 1354

Domain labels:
DomainLabel
1    1149
0     205
Name: count, dtype: int64

Sources:
Source
Lung_CT_Nodule    1149
Natural_CIFAR       90
MRI                 48
Chest_Xray          34
Ultrasound          33
Name: count, dtype: int64

VALIDATION
Total: 305

Domain labels:
DomainLabel
1    237
0     68
Name: count, dtype: int64

Sources:
Source
Lung_CT_Nodule    237
Natural_CIFAR      30
MRI                16
Chest_Xray         11
Ultrasound         11
Name: count, dtype: int64

TEST
Total: 325

Domain labels:
DomainLabel
1    256
0     69
Name: count, dtype: int64

Sources:
Source
Lung_CT_Nodule    256
Natural_CIFAR      30
MRI                16
Chest_Xray         12
Ultrasound         11
Name: count, dtype: int64


In [77]:
train_paths = set(
    domain_train_df[
        "ImagePath"
    ]
)

validation_paths = set(
    domain_validation_df[
        "ImagePath"
    ]
)

test_paths = set(
    domain_test_df[
        "ImagePath"
    ]
)


print(
    "Train ↔ Validation overlap:",
    len(
        train_paths
        & validation_paths
    )
)

print(
    "Train ↔ Test overlap:",
    len(
        train_paths
        & test_paths
    )
)

print(
    "Validation ↔ Test overlap:",
    len(
        validation_paths
        & test_paths
    )
)

Train ↔ Validation overlap: 0
Train ↔ Test overlap: 0
Validation ↔ Test overlap: 0


##13. Save split manifests now

In [78]:
SPLIT_ROOT = (
    DOMAIN_MODEL_ROOT
    / "Splits"
)

SPLIT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


domain_train_df.to_csv(
    SPLIT_ROOT
    / "domain_train.csv",
    index=False
)


domain_validation_df.to_csv(
    SPLIT_ROOT
    / "domain_validation.csv",
    index=False
)


domain_test_df.to_csv(
    SPLIT_ROOT
    / "domain_test.csv",
    index=False
)


print(
    "Saved domain split manifests."
)

Saved domain split manifests.


##14. Image loading - important design choice

In [79]:
IMAGE_SIZE = (
    128,
    128
)


def load_domain_image(
    image_path,
    label
):

    image = tf.io.read_file(
        image_path
    )

    image = tf.image.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image.set_shape(
        [
            None,
            None,
            3
        ]
    )

    image = tf.image.resize(
        image,
        IMAGE_SIZE
    )


    image = tf.image.rgb_to_grayscale(
        image
    )


    image = tf.image.grayscale_to_rgb(
        image
    )


    image = tf.cast(
        image,
        tf.float32
    )


    label = tf.cast(
        label,
        tf.float32
    )


    return (
        image,
        label
    )

##15. Data augmentation

In [80]:
domain_augmentation = (
    tf.keras.Sequential(
        [

            tf.keras.layers.RandomFlip(
                "horizontal"
            ),

            tf.keras.layers.RandomRotation(
                0.03
            ),

            tf.keras.layers.RandomZoom(
                0.05
            ),

            tf.keras.layers.RandomTranslation(
                0.03,
                0.03
            )

        ],
        name="domain_augmentation"
    )
)

##16. Build datasets

In [81]:
BATCH_SIZE = 32

AUTOTUNE = (
    tf.data.AUTOTUNE
)


def build_domain_dataset(
    dataframe,
    training=False
):

    paths = (
        dataframe[
            "ImagePath"
        ]
        .astype(str)
        .values
    )


    labels = (
        dataframe[
            "DomainLabel"
        ]
        .astype(
            np.float32
        )
        .values
    )


    dataset = (
        tf.data.Dataset
        .from_tensor_slices(
            (
                paths,
                labels
            )
        )
    )


    dataset = dataset.map(
        load_domain_image,
        num_parallel_calls=AUTOTUNE
    )


    if training:

        dataset = dataset.shuffle(
            buffer_size=len(
                dataframe
            ),
            seed=SEED
        )


    dataset = dataset.batch(
        BATCH_SIZE
    )


    dataset = dataset.prefetch(
        AUTOTUNE
    )


    return dataset

In [82]:
domain_train_ds = (
    build_domain_dataset(
        domain_train_df,
        training=True
    )
)


domain_validation_ds = (
    build_domain_dataset(
        domain_validation_df
    )
)


domain_test_ds = (
    build_domain_dataset(
        domain_test_df
    )
)

##17. Build MobileNetV2 domain classifier

In [83]:
from tensorflow.keras.applications import (
    MobileNetV2
)


domain_backbone = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(
        IMAGE_SIZE[0],
        IMAGE_SIZE[1],
        3
    )
)


domain_backbone.trainable = False


print(
    "Backbone layers:",
    len(
        domain_backbone.layers
    )
)

Backbone layers: 154


###Models

In [84]:
inputs = tf.keras.Input(
    shape=(
        IMAGE_SIZE[0],
        IMAGE_SIZE[1],
        3
    ),
    name="domain_input"
)


x = domain_augmentation(
    inputs
)


x = (
    tf.keras.applications
    .mobilenet_v2
    .preprocess_input(
        x
    )
)


x = domain_backbone(
    x,
    training=False
)


x = (
    tf.keras.layers
    .GlobalAveragePooling2D(
        name="domain_gap"
    )(
        x
    )
)


x = tf.keras.layers.Dropout(
    0.30,
    name="domain_dropout"
)(
    x
)


outputs = tf.keras.layers.Dense(
    1,
    activation="sigmoid",
    name="domain_output"
)(
    x
)


domain_model = (
    tf.keras.Model(
        inputs=inputs,
        outputs=outputs,
        name="MobileNetV2_Domain_Validator"
    )
)

##18. Compile

In [85]:
domain_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-3
    ),

    loss=tf.keras.losses.BinaryCrossentropy(),

    metrics=[
        tf.keras.metrics.BinaryAccuracy(
            name="accuracy"
        ),

        tf.keras.metrics.Precision(
            name="precision"
        ),

        tf.keras.metrics.Recall(
            name="recall"
        ),

        tf.keras.metrics.AUC(
            name="roc_auc"
        ),

        tf.keras.metrics.AUC(
            curve="PR",
            name="pr_auc"
        )
    ]
)

##19. Class weights

In [86]:
train_labels = (
    domain_train_df[
        "DomainLabel"
    ]
    .astype(int)
    .values
)


classes = np.unique(
    train_labels
)


weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_labels
)


domain_class_weights = {
    int(
        class_label
    ):
        float(
            weight
        )

    for class_label, weight
    in zip(
        classes,
        weights
    )
}


print(
    domain_class_weights
)

{0: 3.3024390243902437, 1: 0.5892080069625761}


##20. Training directories and callbacks

In [87]:
BASELINE_ROOT = (
    DOMAIN_MODEL_ROOT
    / "Baseline"
)

BASELINE_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


BASELINE_BEST = (
    BASELINE_ROOT
    / "best_model.keras"
)

####Callbacks:

In [88]:
domain_callbacks = [

    tf.keras.callbacks.ModelCheckpoint(
        BASELINE_BEST,
        monitor="val_pr_auc",
        mode="max",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_pr_auc",
        mode="max",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )

]

##21. Baseline training

In [89]:
BASELINE_EPOCHS = 20


baseline_history = (
    domain_model.fit(
        domain_train_ds,

        validation_data=
            domain_validation_ds,

        epochs=
            BASELINE_EPOCHS,

        class_weight=
            domain_class_weights,

        callbacks=
            domain_callbacks
    )
)

Epoch 1/20
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.7885 - loss: 0.4814 - pr_auc: 0.9643 - precision: 0.9466 - recall: 0.7967 - roc_auc: 0.8275
Epoch 1: val_pr_auc improved from None to 0.99954, saving model to /content/drive/MyDrive/Dissertation/Models/Domain_Validation/Experiment_02_Supervised/Baseline/best_model.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Dissertation/Models/Domain_Validation/Experiment_02_Supervised/Baseline/best_model.keras
43/43 ━━━━━━━━━━━━━━━━━━━━ 42s 227ms/step - accuracy: 0.8604 - loss: 0.3316 - pr_auc: 0.9874 - precision: 0.9752 - recall: 0.8573 - roc_auc: 0.9323 - val_accuracy: 0.9836 - val_loss: 0.0772 - val_pr_auc: 0.9995 - val_precision: 0.9957 - val_recall: 0.9831 - val_roc_auc: 0.9984 - learning_rate: 0.0010
Epoch 2/20
42/43 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.9416 - loss: 0.1388 - pr_auc: 0.9979 - precision: 0.9907 - recall: 0.9405 - roc_auc: 0.9871
Epoch 2: val_pr_auc improved from 0.99954 to 0.99996, savi

##22. Reload the Best Frozen MobileNetV2 Baseline

The checkpoint with the highest validation PR-AUC is reloaded before
threshold calibration and detailed validation analysis.

All threshold selection is performed using the validation set only.
The independent test set remains untouched until final model selection.

In [90]:
# ============================================================
# Reload Best Baseline Checkpoint
# ============================================================

domain_baseline_best = (
    tf.keras.models.load_model(
        BASELINE_BEST,
        compile=False
    )
)

print(
    "Loaded baseline model:"
)

print(
    BASELINE_BEST
)

print(
    "Input shape:",
    domain_baseline_best.input_shape
)

print(
    "Output shape:",
    domain_baseline_best.output_shape
)

Loaded baseline model:
/content/drive/MyDrive/Dissertation/Models/Domain_Validation/Experiment_02_Supervised/Baseline/best_model.keras
Input shape: (None, 128, 128, 3)
Output shape: (None, 1)


## 23. Generate Baseline Validation Predictions

The domain classifier outputs the probability that an input belongs to
the supported preprocessed lung CT nodule domain.

- Higher probability → more evidence for Supported
- Lower probability → more evidence for Unsupported

In [91]:
validation_domain_prob = (
    domain_baseline_best.predict(
        domain_validation_ds,
        verbose=1
    )
    .ravel()
)


validation_domain_true = (
    domain_validation_df[
        "DomainLabel"
    ]
    .astype(
        np.int32
    )
    .values
)


print(
    "Labels shape:",
    validation_domain_true.shape
)

print(
    "Predictions shape:",
    validation_domain_prob.shape
)

print(
    "Minimum probability:",
    validation_domain_prob.min()
)

print(
    "Maximum probability:",
    validation_domain_prob.max()
)

print(
    "Mean probability:",
    validation_domain_prob.mean()
)

10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 222ms/step
Labels shape: (305,)
Predictions shape: (305,)
Minimum probability: 0.0010985516
Maximum probability: 0.9999478
Mean probability: 0.7828657


##24. Baseline validation metrics at threshold 0.5

In [92]:
BASELINE_DEFAULT_THRESHOLD = (
    0.50
)


validation_pred_05 = (
    validation_domain_prob
    >= BASELINE_DEFAULT_THRESHOLD
).astype(
    np.int32
)


print(
    classification_report(
        validation_domain_true,
        validation_pred_05,

        target_names=[
            "Unsupported",
            "Supported CT"
        ],

        digits=4
    )
)


print(
    confusion_matrix(
        validation_domain_true,
        validation_pred_05
    )
)

              precision    recall  f1-score   support

 Unsupported     1.0000    1.0000    1.0000        68
Supported CT     1.0000    1.0000    1.0000       237

    accuracy                         1.0000       305
   macro avg     1.0000    1.0000    1.0000       305
weighted avg     1.0000    1.0000    1.0000       305

[[ 68   0]
 [  0 237]]


In [93]:
baseline_validation_roc_auc = (
    roc_auc_score(
        validation_domain_true,
        validation_domain_prob
    )
)


baseline_validation_pr_auc = (
    average_precision_score(
        validation_domain_true,
        validation_domain_prob
    )
)


print(
    "Validation ROC-AUC:",
    baseline_validation_roc_auc
)

print(
    "Validation PR-AUC:",
    baseline_validation_pr_auc
)

Validation ROC-AUC: 1.0
Validation PR-AUC: 1.0


## 25. Validation-Based Operating Threshold Selection

The domain validator is a safety gate rather than the final clinical
classifier.

Threshold selection therefore prioritizes retention of legitimate CT
nodule inputs.

The operating threshold is selected as the highest validation threshold
that preserves at least 98% recall for supported CT-nodule images.

Among thresholds satisfying this requirement, a higher threshold is
preferred because it provides stronger rejection of unsupported inputs.

The independent test set is not used during threshold selection.

In [94]:
# ============================================================
# Threshold Search
# ============================================================

TARGET_SUPPORTED_RECALL = (
    0.98
)


threshold_candidates = (
    np.linspace(
        0.01,
        0.99,
        981
    )
)


threshold_rows = []


for threshold in threshold_candidates:

    predicted = (
        validation_domain_prob
        >= threshold
    ).astype(
        np.int32
    )


    # Supported CT = positive class
    supported_recall = (
        recall_score(
            validation_domain_true,
            predicted,
            pos_label=1,
            zero_division=0
        )
    )


    # Unsupported rejection = specificity
    unsupported_mask = (
        validation_domain_true
        == 0
    )

    unsupported_rejection = (
        np.mean(
            predicted[
                unsupported_mask
            ]
            == 0
        )
    )


    threshold_rows.append(
        {
            "Threshold":
                threshold,

            "SupportedRecall":
                supported_recall,

            "UnsupportedRejection":
                unsupported_rejection
        }
    )


threshold_results = (
    pd.DataFrame(
        threshold_rows
    )
)

In [95]:
eligible_thresholds = (
    threshold_results[
        threshold_results[
            "SupportedRecall"
        ]
        >= TARGET_SUPPORTED_RECALL
    ]
)


if eligible_thresholds.empty:

    raise RuntimeError(
        "No threshold satisfies the "
        "required supported CT recall."
    )


DOMAIN_THRESHOLD_BASELINE = float(
    eligible_thresholds
    .sort_values(
        "Threshold",
        ascending=False
    )
    .iloc[0][
        "Threshold"
    ]
)


selected_row = (
    eligible_thresholds[
        eligible_thresholds[
            "Threshold"
        ]
        == DOMAIN_THRESHOLD_BASELINE
    ]
    .iloc[0]
)


print(
    "Selected baseline threshold:",
    DOMAIN_THRESHOLD_BASELINE
)

print(
    "Supported CT recall:",
    selected_row[
        "SupportedRecall"
    ]
)

print(
    "Unsupported rejection:",
    selected_row[
        "UnsupportedRejection"
    ]
)

Selected baseline threshold: 0.897
Supported CT recall: 0.9831223628691983
Unsupported rejection: 1.0


##26. Evaluate the threshold by validation source

In [96]:
validation_analysis = (
    domain_validation_df.copy()
)


validation_analysis[
    "SupportedProbability"
] = validation_domain_prob


validation_analysis[
    "PredictedDomainLabel"
] = (
    validation_domain_prob
    >= DOMAIN_THRESHOLD_BASELINE
).astype(
    np.int32
)


validation_source_summary = (
    validation_analysis
    .groupby(
        "Source"
    )
    .agg(
        Samples=(
            "PredictedDomainLabel",
            "size"
        ),

        MeanProbability=(
            "SupportedProbability",
            "mean"
        ),

        Accepted=(
            "PredictedDomainLabel",
            "sum"
        )
    )
    .reset_index()
)


validation_source_summary[
    "AcceptanceRate"
] = (
    validation_source_summary[
        "Accepted"
    ]
    /
    validation_source_summary[
        "Samples"
    ]
)


validation_source_summary[
    "RejectionRate"
] = (
    1
    -
    validation_source_summary[
        "AcceptanceRate"
    ]
)


display(
    validation_source_summary
)

,Source,Samples,MeanProbability,Accepted,AcceptanceRate,RejectionRate
0,Chest_Xray,11,0.061224,0,0.000000,1.000000
1,Lung_CT_Nodule,237,0.987421,233,0.983122,0.016878
2,MRI,16,0.046092,0,0.000000,1.000000
3,Natural_CIFAR,30,0.106000,0,0.000000,1.000000
4,Ultrasound,11,0.014935,0,0.000000,1.000000


##27. Check benign vs malignant validaton CT retention

In [97]:
ct_validation_analysis = (
    validation_analysis[
        validation_analysis[
            "Source"
        ]
        == "Lung_CT_Nodule"
    ]
    .copy()
)


ct_class_summary = (
    ct_validation_analysis
    .groupby(
        "CancerLabel"
    )
    .agg(
        Samples=(
            "PredictedDomainLabel",
            "size"
        ),

        Accepted=(
            "PredictedDomainLabel",
            "sum"
        ),

        MeanProbability=(
            "SupportedProbability",
            "mean"
        )
    )
    .reset_index()
)


ct_class_summary[
    "Class"
] = (
    ct_class_summary[
        "CancerLabel"
    ]
    .map(
        {
            0: "Benign",
            1: "Malignant"
        }
    )
)


ct_class_summary[
    "AcceptanceRate"
] = (
    ct_class_summary[
        "Accepted"
    ]
    /
    ct_class_summary[
        "Samples"
    ]
)


display(
    ct_class_summary[
        [
            "Class",
            "Samples",
            "Accepted",
            "AcceptanceRate",
            "MeanProbability"
        ]
    ]
)

,Class,Samples,Accepted,AcceptanceRate,MeanProbability
0,Benign,209,207,0.990431,0.988108
1,Malignant,28,26,0.928571,0.982296


## 28. Experiment 2B — Fine-Tune the Final 20 MobileNetV2 Layers

The best frozen-backbone model is fine-tuned conservatively to determine
whether limited adaptation improves separation of visually challenging
unsupported inputs, particularly chest X-rays.

Only the final 20 MobileNetV2 backbone layers are unfrozen.

A substantially lower learning rate is used to avoid destroying
pretrained ImageNet features.

In [98]:
domain_finetune_model = (
    tf.keras.models.load_model(
        BASELINE_BEST,
        compile=False
    )
)

In [99]:
#Find backbone:#

mobilenet_backbone = None


for layer in (
    domain_finetune_model.layers
):

    if (
        isinstance(
            layer,
            tf.keras.Model
        )

        and

        "mobilenet" in
        layer.name.lower()
    ):

        mobilenet_backbone = (
            layer
        )

        break


if mobilenet_backbone is None:

    raise RuntimeError(
        "MobileNetV2 backbone not found."
    )


print(
    "Backbone:",
    mobilenet_backbone.name
)

print(
    "Backbone layers:",
    len(
        mobilenet_backbone.layers
    )
)

Backbone: mobilenetv2_1.00_128
Backbone layers: 154


In [100]:
#Unfreeze: #
mobilenet_backbone.trainable = (
    True
)


for layer in (
    mobilenet_backbone.layers[
        :-20
    ]
):

    layer.trainable = (
        False
    )


for layer in (
    mobilenet_backbone.layers[
        -20:
    ]
):

    # Keep BatchNorm frozen
    if isinstance(
        layer,
        tf.keras.layers.BatchNormalization
    ):

        layer.trainable = (
            False
        )

    else:

        layer.trainable = (
            True
        )


print(
    "Trainable backbone layers:",
    sum(
        layer.trainable
        for layer
        in mobilenet_backbone.layers
    )
)

Trainable backbone layers: 13


##29. Compile fine-tuning model

In [101]:
domain_finetune_model.compile(
    optimizer=(
        tf.keras.optimizers.Adam(
            learning_rate=1e-5
        )
    ),

    loss=(
        tf.keras.losses
        .BinaryCrossentropy()
    ),

    metrics=[
        tf.keras.metrics.BinaryAccuracy(
            name="accuracy"
        ),

        tf.keras.metrics.Precision(
            name="precision"
        ),

        tf.keras.metrics.Recall(
            name="recall"
        ),

        tf.keras.metrics.AUC(
            name="roc_auc"
        ),

        tf.keras.metrics.AUC(
            curve="PR",
            name="pr_auc"
        )
    ]
)

In [102]:
FINETUNE_ROOT = (
    DOMAIN_MODEL_ROOT
    / "FineTune_Last20"
)


FINETUNE_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


FINETUNE_BEST = (
    FINETUNE_ROOT
    / "best_model.keras"
)

####Callbacks

In [103]:
finetune_callbacks = [

    tf.keras.callbacks.ModelCheckpoint(
        FINETUNE_BEST,
        monitor="val_pr_auc",
        mode="max",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_pr_auc",
        mode="max",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.2,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
]

####Train

In [104]:
FINETUNE_EPOCHS = (
    15
)


finetune_history = (
    domain_finetune_model.fit(
        domain_train_ds,

        validation_data=
            domain_validation_ds,

        epochs=
            FINETUNE_EPOCHS,

        class_weight=
            domain_class_weights,

        callbacks=
            finetune_callbacks
    )
)

Epoch 1/15
43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.9692 - loss: 0.0724 - pr_auc: 0.9994 - precision: 0.9929 - recall: 0.9707 - roc_auc: 0.9966
Epoch 1: val_pr_auc improved from None to 1.00000, saving model to /content/drive/MyDrive/Dissertation/Models/Domain_Validation/Experiment_02_Supervised/FineTune_Last20/best_model.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Dissertation/Models/Domain_Validation/Experiment_02_Supervised/FineTune_Last20/best_model.keras
43/43 ━━━━━━━━━━━━━━━━━━━━ 16s 131ms/step - accuracy: 0.9786 - loss: 0.0585 - pr_auc: 0.9995 - precision: 0.9973 - recall: 0.9774 - roc_auc: 0.9972 - val_accuracy: 1.0000 - val_loss: 0.0094 - val_pr_auc: 1.0000 - val_precision: 1.0000 - val_recall: 1.0000 - val_roc_auc: 1.0000 - learning_rate: 1.0000e-05
Epoch 2/15
41/43 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9940 - loss: 0.0282 - pr_auc: 1.0000 - precision: 1.0000 - recall: 0.9929 - roc_auc: 0.9998
Epoch 2: val_pr_auc did not improve from

##30. Reload best fine-tuned model

In [105]:
domain_finetune_best = (
    tf.keras.models.load_model(
        FINETUNE_BEST,
        compile=False
    )
)

##31. Fine-tuned validation predictions

In [106]:
validation_prob_finetuned = (
    domain_finetune_best.predict(
        domain_validation_ds,
        verbose=1
    )
    .ravel()
)

10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 172ms/step


##31.1 Select Fine-Tuned Threshold

In [108]:
# ============================================================
# Select Fine-Tuned Validation Threshold
# ============================================================

TARGET_SUPPORTED_RECALL = 0.98

threshold_candidates = np.linspace(
    0.01,
    0.99,
    981
)

rows = []

for threshold in threshold_candidates:

    predicted = (
        validation_prob_finetuned >= threshold
    ).astype(np.int32)

    supported_recall = recall_score(
        validation_domain_true,
        predicted,
        pos_label=1,
        zero_division=0
    )

    unsupported_mask = (
        validation_domain_true == 0
    )

    unsupported_rejection = np.mean(
        predicted[unsupported_mask] == 0
    )

    rows.append({

        "Threshold": threshold,

        "SupportedRecall": supported_recall,

        "UnsupportedRejection": unsupported_rejection

    })

threshold_results_finetuned = pd.DataFrame(rows)

In [109]:
eligible = threshold_results_finetuned[
    threshold_results_finetuned[
        "SupportedRecall"
    ] >= TARGET_SUPPORTED_RECALL
]

DOMAIN_THRESHOLD_FINETUNED = float(

    eligible
    .sort_values(
        "Threshold",
        ascending=False
    )
    .iloc[0]["Threshold"]

)

print(
    "Fine-tuned threshold:",
    DOMAIN_THRESHOLD_FINETUNED
)

Fine-tuned threshold: 0.9680000000000001


####Then evaluate the fine-tuned model using that threshold

In [113]:
validation_analysis_finetuned = (
    domain_validation_df.copy()
)

validation_analysis_finetuned[
    "Probability"
] = validation_prob_finetuned

validation_analysis_finetuned[
    "Prediction"
] = (
    validation_prob_finetuned
    >= DOMAIN_THRESHOLD_FINETUNED
).astype(np.int32)

####Finally generate the comparison table

In [114]:
comparison = pd.DataFrame(
[
    {
        "Model":"Frozen MobileNetV2",
        "Validation ROC-AUC":baseline_validation_roc_auc,
        "Validation PR-AUC":baseline_validation_pr_auc,
        "Threshold":DOMAIN_THRESHOLD_BASELINE
    },

    {
        "Model":"Fine-Tuned MobileNetV2",
        "Validation ROC-AUC":roc_auc_score(
            validation_domain_true,
            validation_prob_finetuned
        ),
        "Validation PR-AUC":average_precision_score(
            validation_domain_true,
            validation_prob_finetuned
        ),
        "Threshold":DOMAIN_THRESHOLD_FINETUNED
    }
]
)

display(comparison)

,Model,Validation ROC-AUC,Validation PR-AUC,Threshold
0,Frozen MobileNetV2,1.0,1.0,0.897
1,Fine-Tuned MobileNetV2,1.0,1.0,0.968


##32 Compare Frozen vs fine-tuned validation results

In [117]:
# ============================================================
# Frozen Validation Metrics
# ============================================================

baseline_validation_roc_auc = roc_auc_score(
    validation_domain_true,
    validation_domain_prob
)

baseline_validation_pr_auc = average_precision_score(
    validation_domain_true,
    validation_domain_prob
)

# ============================================================
# Fine-Tuned Validation Metrics
# ============================================================

finetuned_validation_roc_auc = roc_auc_score(
    validation_domain_true,
    validation_prob_finetuned
)

finetuned_validation_pr_auc = average_precision_score(
    validation_domain_true,
    validation_prob_finetuned
)

In [118]:
comparison = pd.DataFrame({

    "Model":[
        "Frozen MobileNetV2",
        "Fine-Tuned MobileNetV2"
    ],

    "Validation ROC-AUC":[
        baseline_validation_roc_auc,
        finetuned_validation_roc_auc
    ],

    "Validation PR-AUC":[
        baseline_validation_pr_auc,
        finetuned_validation_pr_auc
    ],

    "Threshold":[
        DOMAIN_THRESHOLD_BASELINE,
        DOMAIN_THRESHOLD_FINETUNED
    ]

})

display(comparison)

,Model,Validation ROC-AUC,Validation PR-AUC,Threshold
0,Frozen MobileNetV2,1.0,1.0,0.897
1,Fine-Tuned MobileNetV2,1.0,1.0,0.968


## 33. Select Final Domain Validator

Both the frozen and fine-tuned MobileNetV2 domain validators are compared
using the validation set.

The final model is selected before evaluating the independent test set.

Selection priority:

1. Supported CT retention
2. Malignant CT retention
3. Unsupported image rejection
4. Validation ROC-AUC
5. Validation PR-AUC

In [119]:
if finetuned_validation_pr_auc >= baseline_validation_pr_auc:

    FINAL_DOMAIN_MODEL = domain_finetune_best

    FINAL_DOMAIN_THRESHOLD = DOMAIN_THRESHOLD_FINETUNED

    FINAL_MODEL_NAME = "Fine-Tuned MobileNetV2"

else:

    FINAL_DOMAIN_MODEL = domain_baseline_best

    FINAL_DOMAIN_THRESHOLD = DOMAIN_THRESHOLD_BASELINE

    FINAL_MODEL_NAME = "Frozen MobileNetV2"

print("Selected:", FINAL_MODEL_NAME)

Selected: Fine-Tuned MobileNetV2


##34. Independent Test Evaluation

In [120]:
# ============================================================
# Generate Test Predictions
# ============================================================

test_probabilities = (
    FINAL_DOMAIN_MODEL.predict(
        domain_test_ds,
        verbose=1
    ).ravel()
)

test_true = (
    domain_test_df["DomainLabel"]
    .astype(np.int32)
    .values
)

test_predictions = (
    test_probabilities
    >= FINAL_DOMAIN_THRESHOLD
).astype(np.int32)

# ============================================================
# Metrics
# ============================================================

test_roc_auc = roc_auc_score(
    test_true,
    test_probabilities
)

test_pr_auc = average_precision_score(
    test_true,
    test_probabilities
)

print("Test ROC-AUC:", test_roc_auc)
print("Test PR-AUC:", test_pr_auc)

print(
    classification_report(
        test_true,
        test_predictions,
        target_names=[
            "Unsupported",
            "Supported CT"
        ],
        digits=4
    )
)

cm = confusion_matrix(
    test_true,
    test_predictions
)

print(cm)

11/11 ━━━━━━━━━━━━━━━━━━━━ 140s 12s/step
Test ROC-AUC: 1.0
Test PR-AUC: 1.0
              precision    recall  f1-score   support

 Unsupported     0.9200    1.0000    0.9583        69
Supported CT     1.0000    0.9766    0.9881       256

    accuracy                         0.9815       325
   macro avg     0.9600    0.9883    0.9732       325
weighted avg     0.9830    0.9815    0.9818       325

[[ 69   0]
 [  6 250]]


##35. per-Source Test Evaluation

In [121]:
test_analysis = domain_test_df.copy()

test_analysis["Probability"] = test_probabilities

test_analysis["Prediction"] = test_predictions

source_summary = (

    test_analysis

    .groupby("Source")

    .agg(

        Samples=("Prediction","size"),

        Accepted=("Prediction","sum"),

        MeanProbability=("Probability","mean")

    )

    .reset_index()

)

source_summary["AcceptanceRate"] = (

    source_summary["Accepted"]

    / source_summary["Samples"]

)

source_summary["RejectionRate"] = (

    1

    - source_summary["AcceptanceRate"]

)

display(source_summary)

,Source,Samples,Accepted,MeanProbability,AcceptanceRate,RejectionRate
0,Chest_Xray,12,0,0.005835,0.000000,1.000000
1,Lung_CT_Nodule,256,250,0.995345,0.976562,0.023438
2,MRI,16,0,0.009635,0.000000,1.000000
3,Natural_CIFAR,30,0,0.038181,0.000000,1.000000
4,Ultrasound,11,0,0.001159,0.000000,1.000000


##36. Benign vs Malignant

In [122]:
ct_only = test_analysis[
    test_analysis["Source"]=="Lung_CT_Nodule"
].copy()

class_summary = (

    ct_only

    .groupby("CancerLabel")

    .agg(

        Samples=("Prediction","size"),

        Accepted=("Prediction","sum"),

        MeanProbability=("Probability","mean")

    )

    .reset_index()

)

class_summary["Class"] = class_summary[
    "CancerLabel"
].map({

    0:"Benign",

    1:"Malignant"

})

class_summary["AcceptanceRate"] = (

    class_summary["Accepted"]

    / class_summary["Samples"]

)

display(class_summary)

,CancerLabel,Samples,Accepted,MeanProbability,Class,AcceptanceRate
0,0.0,221,216,0.995251,Benign,0.977376
1,1.0,35,34,0.995941,Malignant,0.971429


##37. Error Analysis

####Rejection legitimate CT:

In [123]:
false_rejected = test_analysis[
    (test_analysis["Source"]=="Lung_CT_Nodule")
    &
    (test_analysis["Prediction"]==0)
]

display(false_rejected)

,ImagePath,Source,CancerLabel,PatientID,Split,DomainLabel,Probability,Prediction
15,/content/drive/MyDrive/Dissertation/Dataset/Pr...,Lung_CT_Nodule,1.0,LIDC-IDRI-0077,test,1,0.919257,0
38,/content/drive/MyDrive/Dissertation/Dataset/Pr...,Lung_CT_Nodule,0.0,LIDC-IDRI-0147,test,1,0.964708,0
119,/content/drive/MyDrive/Dissertation/Dataset/Pr...,Lung_CT_Nodule,0.0,LIDC-IDRI-0356,test,1,0.953681,0
202,/content/drive/MyDrive/Dissertation/Dataset/Pr...,Lung_CT_Nodule,0.0,LIDC-IDRI-0795,test,1,0.954860,0
227,/content/drive/MyDrive/Dissertation/Dataset/Pr...,Lung_CT_Nodule,0.0,LIDC-IDRI-0910,test,1,0.955067,0
254,/content/drive/MyDrive/Dissertation/Dataset/Pr...,Lung_CT_Nodule,0.0,LIDC-IDRI-0814,test,1,0.963374,0


Accepted OOD:

In [124]:
false_accepted = test_analysis[
    (test_analysis["Source"]!="Lung_CT_Nodule")
    &
    (test_analysis["Prediction"]==1)
]

display(false_accepted)

,ImagePath,Source,CancerLabel,PatientID,Split,DomainLabel,Probability,Prediction


##38. Export

In [125]:
FINAL_EXPORT = DOMAIN_MODEL_ROOT/"Final"

FINAL_EXPORT.mkdir(
    parents=True,
    exist_ok=True
)

FINAL_DOMAIN_MODEL.save(

    FINAL_EXPORT/"best_model.keras"

)

summary = {

    "SelectedModel":FINAL_MODEL_NAME,

    "Threshold":float(FINAL_DOMAIN_THRESHOLD),

    "TestROC_AUC":float(test_roc_auc),

    "TestPR_AUC":float(test_pr_auc)

}

with open(

    FINAL_EXPORT/"domain_threshold.json",

    "w"

) as f:

    json.dump(

        summary,

        f,

        indent=4

    )

source_summary.to_csv(

    FINAL_EXPORT/"test_source_summary.csv",

    index=False

)

class_summary.to_csv(

    FINAL_EXPORT/"test_ct_summary.csv",

    index=False

)

print("Final domain validator exported.")

Final domain validator exported.


##39. Final Experiment Summary

In [126]:
print("="*70)

print("FINAL DOMAIN VALIDATOR")

print("="*70)

print("Model:",FINAL_MODEL_NAME)

print("Threshold:",FINAL_DOMAIN_THRESHOLD)

print("Test ROC-AUC:",test_roc_auc)

print("Test PR-AUC:",test_pr_auc)

print("\nPer-source summary")

display(source_summary)

print("\nCT acceptance")

display(class_summary)

FINAL DOMAIN VALIDATOR
Model: Fine-Tuned MobileNetV2
Threshold: 0.9680000000000001
Test ROC-AUC: 1.0
Test PR-AUC: 1.0

Per-source summary


,Source,Samples,Accepted,MeanProbability,AcceptanceRate,RejectionRate
0,Chest_Xray,12,0,0.005835,0.000000,1.000000
1,Lung_CT_Nodule,256,250,0.995345,0.976562,0.023438
2,MRI,16,0,0.009635,0.000000,1.000000
3,Natural_CIFAR,30,0,0.038181,0.000000,1.000000
4,Ultrasound,11,0,0.001159,0.000000,1.000000



CT acceptance


,CancerLabel,Samples,Accepted,MeanProbability,Class,AcceptanceRate
0,0.0,221,216,0.995251,Benign,0.977376
1,1.0,35,34,0.995941,Malignant,0.971429
